# 02 · Descargar ortomosaicos de WeedsGalore sin usar Drive

Este notebook descarga el ZIP de ortomosaicos en `/content` y permite descargar al computador los `.tif` extraídos.

Importante: los ortomosaicos son pesados. Si la descarga desde Colab falla, se recomienda descargarlos directamente en el computador usando PowerShell.

In [ ]:
from pathlib import Path
import zipfile
import shutil
import subprocess

BASE_DIR = Path('/content/proyecto_malezas/datasets/weedsgalore')
DOWNLOAD_DIR = BASE_DIR / 'descargas'
ORTHO_DIR = BASE_DIR / 'weedsgalore-orthomosaic'

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
ORTHO_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)

In [ ]:
ORTHOMOSAIC_URL = 'https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-orthomosaic.zip'
ZIP_PATH = DOWNLOAD_DIR / 'weedsgalore-orthomosaic.zip'

if ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0:
    print('El ZIP ya existe:', ZIP_PATH)
else:
    subprocess.run(['wget', '-c', '-O', str(ZIP_PATH), ORTHOMOSAIC_URL], check=True)

print('ZIP:', ZIP_PATH)
print('Tamaño GB:', ZIP_PATH.stat().st_size / (1024 ** 3))

In [ ]:
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    nombres = z.namelist()

print('Archivos dentro del ZIP:')
for n in nombres[:50]:
    print(n)
print('Total:', len(nombres))

In [ ]:
EXTRAER_TODOS = False
archivos_recomendados = ['2023-06-06_om.tif', '2023-05-30_om.tif']

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    if EXTRAER_TODOS:
        print('Extrayendo todos los archivos...')
        z.extractall(ORTHO_DIR)
    else:
        for objetivo in archivos_recomendados:
            coincidencias = [n for n in z.namelist() if n.endswith(objetivo)]
            if not coincidencias:
                print('No se encontró:', objetivo)
                continue
            for nombre_zip in coincidencias:
                print('Extrayendo:', nombre_zip)
                destino = ORTHO_DIR / Path(nombre_zip).name
                with z.open(nombre_zip) as src, open(destino, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                print('Guardado en:', destino)
                print('Tamaño GB:', destino.stat().st_size / (1024 ** 3))
print('Extracción terminada.')

In [ ]:
archivos_extraidos = sorted([archivo for archivo in ORTHO_DIR.glob('*') if archivo.is_file() and archivo.suffix.lower() in ['.tif', '.tiff']])
print('Archivos GeoTIFF extraídos:')
for archivo in archivos_extraidos:
    print(f'{archivo.name} - {archivo.stat().st_size / (1024 ** 3):.2f} GB')

In [ ]:
from google.colab import files

# Estos archivos son grandes. La descarga desde Colab puede tardar o fallar.
# Si falla, usa la opción PowerShell de la siguiente celda de texto.
DESCARGAR_A_PC = True

if DESCARGAR_A_PC:
    for archivo in archivos_extraidos:
        print('Descargando al computador:', archivo.name)
        files.download(str(archivo))
else:
    print('DESCARGAR_A_PC está en False.')

## Alternativa recomendada: descarga directa en Windows

Para archivos grandes suele ser más estable descargar directamente al computador con PowerShell:

```powershell
mkdir "C:\Vision Computacional\datasets\weedsgalore\weedsgalore-orthomosaic"

curl.exe -L -C - -o "C:\Vision Computacional\datasets\weedsgalore\weedsgalore-orthomosaic\weedsgalore-orthomosaic.zip" "https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-orthomosaic.zip"
```

Luego descomprime el ZIP con 7-Zip o WinRAR y usa en la app la ruta local del `.tif`.